# Average utilization and Figure 8

Average CPU, memory and GPU utilization per container and the GPU-enabled containers with 0% GPU utilization (Figure 8). Run by `run_manuscript.py`; outputs in `05_Results/section4/execution_runs/02_utilization`.

## Setup

In [ ]:
# Locate the repository root and load the folder constants.
from pathlib import Path
import sys
_start = Path.cwd().resolve()
_root = next((p for p in [_start,*_start.parents] if (p/'project_paths.py').is_file()), None)
if _root is None:
    raise RuntimeError('Run from inside the repository.')
for _p in [_root,_root/'04_Analysis/revision',_root/'02_Preprocessing']:
    sys.path.insert(0,str(_p))
from project_paths import *
install_data_io_guard()
PROJECT_ROOT = REVISION_CODE
ensure_dirs()
# Execution scope and output capture (no change to policy calculations).
import json as _mf_json
import re as _mf_re
import numpy as _mf_np
import pandas as _mf_pd
import matplotlib.pyplot as _mf_plt
from matplotlib.figure import Figure as _mf_Figure
_MF_OUTPUT = ROOT / '05_Results/section4/execution_runs/02_utilization'
_MF_OUTPUT.mkdir(parents=True,exist_ok=True)
_MF_CAPTURE = _MF_OUTPUT / 'capture'
_MF_CAPTURE.mkdir(exist_ok=True)
_MF_MODE = 'corrected'
_MF_CELL = 0
_MF_ERRORS = []
PROCESSED_DATA = ROOT / '03_ProcessedData'
RESULTS_SECTION3 = _MF_OUTPUT
RESULTS_SECTION4 = _MF_OUTPUT
RESULTS_REVISION = _MF_OUTPUT
_MF_REAL_READ_CSV = _mf_pd.read_csv
def _mf_read_csv(*args,**kwargs):
    try:
        value = _MF_REAL_READ_CSV(*args,**kwargs)
    except UnicodeDecodeError:
        # Encoding fallback for cp949 CSV files (I/O only).
        value = _MF_REAL_READ_CSV(*args,**{**kwargs,"encoding":"cp949"})
    return value
_mf_pd.read_csv = _mf_read_csv
_mf_orig_savefig = _mf_Figure.savefig
def _mf_savefig(self,fname,*args,**kwargs):
    if isinstance(fname,(str,Path)):
        target=Path(fname)
        if not target.is_absolute() or not target.resolve().is_relative_to(_MF_OUTPUT):
            target = _MF_OUTPUT / target.name
        target.parent.mkdir(parents=True,exist_ok=True)
        fname=target
    return _mf_orig_savefig(self,fname,*args,**kwargs)
_mf_Figure.savefig = _mf_savefig
def _mf_capture_cell(cell_number):
    import warnings as _mf_warnings
    with _mf_warnings.catch_warnings():
        _mf_warnings.simplefilter('ignore')
        for name,value in list(globals().items()):
            if name.startswith('_'):continue
            if isinstance(value,_mf_pd.DataFrame) and len(value) <= 1000 and len(value.columns) <= 80:
                value.to_csv(_MF_CAPTURE/f'cell{cell_number:02d}_{name}.csv',index=True)
            elif isinstance(value,_mf_pd.Series) and len(value) <= 1000:
                value.to_frame().to_csv(_MF_CAPTURE/f'cell{cell_number:02d}_{name}.csv',index=True)
        for no in _mf_plt.get_fignums():
            figure=_mf_plt.figure(no)
            if not figure.axes:continue
            path=_MF_CAPTURE/f'cell{cell_number:02d}_figure{no}.png'
            _mf_orig_savefig(figure,path,dpi=110,bbox_inches='tight')
            lines=[]
            for axno,axis in enumerate(figure.axes):
                for lineno,line in enumerate(axis.get_lines()):
                    x,y=line.get_data()
                    if len(x)!=len(y) or len(x)>100000:continue
                    if len(x):
                        lines.extend((axno,lineno,line.get_label(),i,a,b) for i,(a,b) in enumerate(zip(x,y)))
            if lines:
                _mf_pd.DataFrame(lines,columns=['axis','line','label','point','x','y']).to_csv(path.with_suffix('.csv'),index=False)


# Native Matplotlib export; no plot or calculation changes.
_MF_ORIG_SHOW=_mf_plt.show
_MF_FIG_COUNTER={}
def _mf_show(*args,**kwargs):
    for _number in _mf_plt.get_fignums():
        _figure=_mf_plt.figure(_number)
        if not _figure.axes: continue
        _MF_FIG_COUNTER[_MF_CELL]=_MF_FIG_COUNTER.get(_MF_CELL,0)+1
        _stem=f'cell{_MF_CELL:02d}_plot{_MF_FIG_COUNTER[_MF_CELL]:02d}'
        _folder=_MF_OUTPUT/'figures'; _folder.mkdir(exist_ok=True)
        _mf_orig_savefig(_figure,_folder/(_stem+'.png'),dpi=150,bbox_inches='tight')
        _mf_orig_savefig(_figure,_folder/(_stem+'.pdf'),bbox_inches='tight')
    return _MF_ORIG_SHOW(*args,**kwargs)
_mf_plt.show=_mf_show


In [ ]:
_MF_CELL=2
# # 구글드라이브 연결
# from google.colab import drive



In [ ]:
_mf_capture_cell(2)


In [ ]:
_MF_CELL=3
from pathlib import Path
import os

# =========================
# 0) 환경 감지
# =========================
def is_colab():
    return Path("/content").exists() and any(
        k in os.environ for k in ["COLAB_GPU", "COLAB_RELEASE_TAG"]
    )

# =========================
# 1) BASE 경로 설정 (여기만 본인 환경에 맞게)
# =========================
# Colab에서 사용할 "논문투고용_2026년" 루트 폴더
COLAB_BASE = Path(str(PROCESSED_DATA))

# 로컬에서 사용할 루트 폴더 (원하는 위치로 바꾸세요)
LOCAL_BASE = Path(str(PROCESSED_DATA))

BASE = COLAB_BASE if is_colab() else LOCAL_BASE

# Colab인데 드라이브가 아직 마운트 안 됐으면 안내
if is_colab() and not Path("/content/gdrive").exists():
    print("⚠️ Colab에서 Google Drive가 마운트되지 않았습니다.")
    print("   먼저 아래를 실행하세요:")
    print("   from google.colab import drive; drive.mount('/content/gdrive')")

# =========================
# 2) 평균 사용율(Avg Utilization) 분석용 경로 모음
#    - 아래 폴더명은 당신의 실제 폴더 구조에 맞게 1번만 맞추면 됨
# =========================
# P = {
#     # (예시) 평균 사용율 분석에 쓰는 원본 데이터 폴더/파일
#     # 필요에 따라 이름만 바꾸고, 코드는 P[...]로만 접근하세요.
#     "avgutil_dir": BASE / "03. Data_AvgUtil",          # 예: 평균사용률 데이터 폴더(새로 만든 폴더)
#     "avgutil_out_dir": BASE / "03. Output_AvgUtil",    # 결과 저장 폴더
#     "avgutil_summary_csv": BASE / "avg_util_summary.csv",
# }

# # 출력 폴더는 자동 생성
# P["avgutil_out_dir"].mkdir(parents=True, exist_ok=True)

# print("BASE =", BASE)
# for k, v in P.items():
#     print(f"{k}: {v}")

# # (선택) 입력 폴더 존재 체크
# print("\n[CHECK]")
# print("avgutil_dir exists:", P["avgutil_dir"].exists())


In [ ]:
_mf_capture_cell(3)


In [ ]:
_MF_CELL=4
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
default_colors = plt.rcParams['axes.prop_cycle'].by_key()['color'] # Matplotlib 기본 색상 가져오기

from datetime import datetime
import time
import re
from functools import reduce

import warnings
warnings.filterwarnings('ignore')
import matplotlib.cbook

import os

plt.rcParams['font.family'] = 'sans-serif'  # 기본 sans-serif 폰트 사용
plt.rcParams['font.sans-serif'] = ['Arial', 'Liberation Sans', 'DejaVu Sans']  # 대체 폰트
plt.rcParams['axes.unicode_minus'] = False  # 음수 기호 문제 해결


In [ ]:
_mf_capture_cell(4)


In [ ]:
_MF_CELL=5
try:
    df_selected_2 = pd.read_csv(BASE / "UsageLog20230401_20251130.csv", encoding="utf-8")
except UnicodeDecodeError:
    df_selected_2 = pd.read_csv(BASE / "UsageLog20230401_20251130.csv", encoding="cp949")

# Restrict to the common monitored cohort; the Figure 7/8 code is unchanged.
from project_paths import ROOT as _cohort_root
import sys as _cohort_sys
_cohort_sys.path.insert(0,str(_cohort_root/'04_Analysis/section4'))
from common_monitored_cohort import restrict_to_monitored
_df_selected_before_cohort_n=len(df_selected_2)
df_selected_2=restrict_to_monitored(df_selected_2)
print(f'[Common monitored cohort] {_df_selected_before_cohort_n} -> {len(df_selected_2)} containers')


In [ ]:
_mf_capture_cell(5)


In [ ]:
_MF_CELL=6
# 폰트 사이즈 조절
plt.rcParams['font.size'] = 15  # 모든 글자의 크기를 동일하게 설정
plt.rcParams['axes.titlesize'] = 15  # 그래프 제목
plt.rcParams['axes.labelsize'] = 15  # 축 레이블
plt.rcParams['xtick.labelsize'] = 15  # x축 눈금
plt.rcParams['ytick.labelsize'] = 15  # y축 눈금
plt.rcParams['lines.linewidth'] = 3   # 선 그래프 굵기 조절


In [ ]:
_mf_capture_cell(6)


# FIGURE 7 (original version; the paper's Figure 7 is produced by Run00_Figure7.ipynb)
## 0401 리소스 사용율별 컨테이너 수 CDF 그래프
## 0401_resource_utilization_container_count_cdf


In [ ]:
_MF_CELL=8
# Removing entries with missing 'gpu_allocated(group)'
df_selected_2 = df_selected_2[df_selected_2['gpu_allocated(group)'].notna()]

# 컬럼 값 변경
df_selected_2 = df_selected_2.rename(columns={
    "cpu_util_avg(%)": "CPU",
    "mem_util_avg(%)": "Memory",
    "gpu_util_avg_normalized(%)": "GPU"
})

# gpu_allocated(group) 값 변경
df_selected_2['gpu_allocated(group)'] = df_selected_2['gpu_allocated(group)'].replace({
    '0.0_GPU': '0 GPU',
    '0.x_GPU': '0.x GPU',
    '1.0_GPU': '1 GPU',
    '2.0_GPU': '2 GPU',
    '3.0_GPU': '3 GPU',
    '4.0_GPU': '4 GPU'
})

# 0.x GPU 제외 (전체 분석에서 완전 배제. 다수 컨테이너가 GPU를 나눠쓰는 경우, 정확한 컨테이너 별 자원 사용율 확인 불가능)
df_selected_2 = df_selected_2[df_selected_2['gpu_allocated(group)'] != '0.x GPU']


# Splitting the data into GPU used and not used based on 'gpu_allocated(group)'
gpu_used_data = df_selected_2[df_selected_2['gpu_allocated(group)'] != '0 GPU']
gpu_not_used_data = df_selected_2[df_selected_2['gpu_allocated(group)'] == '0 GPU']


# Defining the thresholds for 5% intervals
thresholds_5 = list(range(0, 101, 5))
resources = ["CPU", "Memory", "GPU"]
resources_without_gpu = ["CPU", "Memory"]

# 숫자형 변환 + 범위 정리(선택)
for r in resources:
    df_selected_2[r] = pd.to_numeric(df_selected_2[r], errors="coerce")
    df_selected_2[r] = df_selected_2[r].clip(lower=0, upper=100)

# GPU 사용/미사용 분리
gpu_used_data = df_selected_2[df_selected_2['gpu_allocated(group)'] != '0 GPU']
gpu_not_used_data = df_selected_2[df_selected_2['gpu_allocated(group)'] == '0 GPU']

# used_time 필터 기준(1분 미만은 제외)
MIN_USED_TIME = 1

# 숫자형 변환(안전)
df_selected_2["used_time(minute)"] = pd.to_numeric(df_selected_2["used_time(minute)"], errors="coerce")

# GPU used / not used 먼저 분리
gpu_used_data = df_selected_2[df_selected_2['gpu_allocated(group)'] != '0 GPU']
gpu_not_used_data = df_selected_2[df_selected_2['gpu_allocated(group)'] == '0 GPU']

# ✅ used_time(minute) < 1 또는 NaN 제거
gpu_used_data = gpu_used_data[gpu_used_data["used_time(minute)"].ge(MIN_USED_TIME)]
gpu_not_used_data = gpu_not_used_data[gpu_not_used_data["used_time(minute)"].ge(MIN_USED_TIME)]


def cdf_props(series, thresholds):
    s = series.dropna()
    if len(s) == 0:
        return [np.nan] * len(thresholds)
    return [s.le(t).mean() for t in thresholds]  # count/len(s)

# 1) GPU-used vs Non-GPU (CPU, Memory)
proportions_5_gpu_used = {}
proportions_5_gpu_not_used = {}

for r in resources_without_gpu:
    proportions_5_gpu_used[r] = cdf_props(gpu_used_data[r], thresholds_5)
    proportions_5_gpu_not_used[r] = cdf_props(gpu_not_used_data[r], thresholds_5)

# 2) GPU utilization: gpu_used만
proportions_5_gpu_used["GPU"] = cdf_props(gpu_used_data["GPU"], thresholds_5)

# 3) ✅ GPU 그룹별(0.x/1/2/3/4)도 동일하게 NaN 제외
groups_sorted = np.sort(gpu_used_data['gpu_allocated(group)'].unique())
proportions_5_by_gpu_group = {g: {} for g in groups_sorted}

for g in groups_sorted:
    gd = gpu_used_data[gpu_used_data['gpu_allocated(group)'] == g]
    for r in resources:
        proportions_5_by_gpu_group[g][r] = cdf_props(gd[r], thresholds_5)

# (선택) 유효 표본수 출력
print("[Valid N] Non-GPU:", len(gpu_not_used_data), "| GPU-used:", len(gpu_used_data))
for r in resources:
    print(
        r,
        "valid_n(non_gpu):", (gpu_not_used_data[r].notna().sum() if r != "GPU" else "-"),
        "| valid_n(gpu_used):", gpu_used_data[r].notna().sum()
    )
for g in groups_sorted:
    gd = gpu_used_data[gpu_used_data['gpu_allocated(group)'] == g]
    print(f"[{g}] valid_n CPU/Mem/GPU:", gd["CPU"].notna().sum(), gd["Memory"].notna().sum(), gd["GPU"].notna().sum())

print("1분 미만 실행된 경우, 값이  Nan 으로 되어, 숫자가 다르게 나온다.")


In [ ]:
_mf_capture_cell(8)


In [ ]:
_MF_CELL=9
fig, axes = plt.subplots(1, 3, figsize=(30, 8))
resources = ["CPU", "Memory", "GPU"]

# 색상/마커 설정
colors = ['tab:blue','tab:orange','tab:green','tab:red','tab:purple','tab:brown','tab:pink']
markers = ['o'] * len(colors)

for idx, resource_to_plot in enumerate(resources):
    ax = axes[idx]

    # --- GPU Not Used ---
    if resource_to_plot != "GPU":
        ax.plot(
            thresholds_5,
            proportions_5_gpu_not_used[resource_to_plot],
            label='Non-GPU',
            color=colors[0],
            marker=markers[0],
            markersize=10,
            linewidth=3
        )

    # --- GPU Used ---
    ax.plot(
        thresholds_5,
        proportions_5_gpu_used[resource_to_plot],
        label='GPU-Enabled AVG.',
        color=colors[1],
        marker=markers[1],
        markersize=10,
        linewidth=3
    )

    # --- GPU 그룹들 ---
    groups_sorted = np.sort(gpu_used_data['gpu_allocated(group)'].unique())
    for i, grp in enumerate(groups_sorted, start=2):
        props = proportions_5_by_gpu_group[grp][resource_to_plot]
        ax.plot(
            thresholds_5,
            props,
            label=grp,
            color=colors[i % len(colors)],
            marker=markers[i % len(markers)],
            markersize=5,
            linewidth=1,
            linestyle='--',
            alpha=0.8
        )

    ax.set_title(f"{resource_to_plot}", fontsize=35)
    ax.set_xlabel("Utilization (%)", fontsize=35)
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 1.2)
    ax.set_xticks(range(0, 101, 20))
    ax.tick_params(axis='x', labelsize=35)
    ax.tick_params(axis='y', labelsize=35)

    # --- y축 레이블 및 눈금 설정 ---
    if idx == 0:
        ax.set_ylabel("Cumulative Fraction of Containers", fontsize=30)
        ax.legend(loc='lower right', fontsize=28)
    else:
        ax.set_yticklabels([])  # y축 눈금 숫자만 제거

    ax.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
_mf_capture_cell(9)


In [ ]:
_MF_CELL=10
# CDF가 1.0보다 작을경우 확인해보는 코드

for r in ["CPU","Memory","GPU"]:
    miss = (~gpu_used_data[r].le(100)).mean()  # NaN 포함해서 le(100)=False면 miss로 잡힘
    print(r, "CDF<1 원인 비율(>100 or NaN):", miss)


In [ ]:
_mf_capture_cell(10)


In [ ]:
_MF_CELL=11
# P50/P90 + ≤5/≤10 (요약표용)

def quick_stats(df, col, label):
    s = df[col].dropna()
    if len(s) == 0:
        print(label, col, "no data")
        return
    print(f"{label:10s} {col:6s} | n={len(s):,} | "
          f"P50={np.percentile(s,50):.1f} | P90={np.percentile(s,90):.1f} | "
          f"<=5%={(s<=5).mean():.2%} | <=10%={(s<=10).mean():.2%}")

for r in ["CPU","Memory","GPU"]:
    if r != "GPU":
        quick_stats(gpu_not_used_data, r, "Non-GPU")
    quick_stats(gpu_used_data, r, "GPU-used")


In [ ]:
_mf_capture_cell(11)


In [ ]:
_MF_CELL=12
df_selected_2['created_time'] = pd.to_datetime(df_selected_2['created_time'])
df_selected_2['terminated_time'] = pd.to_datetime(df_selected_2['terminated_time'])
df_selected_2['run_time'] = pd.to_datetime(df_selected_2['run_time'])
df_selected_2['used_time(minute)'] = (df_selected_2['terminated_time'] - df_selected_2['run_time']).dt.total_seconds() / 60  # execution start to termination

# df_selected_GPU0: GPU allocated인데, 사용율 0인 컨테이너
df_selected_GPU0 = df_selected_2[['gpu_allocated(group)', 'CPU', 'GPU', 'Memory','created_time','terminated_time','used_time(minute)']].copy()
df_selected_GPU0 = df_selected_GPU0[df_selected_GPU0['GPU'] == 0]
df_selected_GPU0 = df_selected_GPU0[df_selected_GPU0['gpu_allocated(group)'] != '0 GPU']

df_selected_GPU0[df_selected_GPU0['gpu_allocated(group)'] == '0 GPU']

df_selected_GPU0['gpu_allocated(group)'].value_counts()


In [ ]:
_mf_capture_cell(12)


Runtime-weighted CDF (그래프용)

참고용


In [ ]:
_MF_CELL=14
# import numpy as np
# import pandas as pd

# # -----------------------------
# # Config
# # -----------------------------
# TIME_COL = "used_time(minute)"
# MIN_USED_TIME = 1
# thresholds_5 = list(range(0, 101, 5))
# resources = ["CPU", "Memory", "GPU"]
# resources_without_gpu = ["CPU", "Memory"]

# # -----------------------------
# # Prep (numeric + filter)
# # -----------------------------
# # 숫자 변환
# for r in resources:
#     df_selected_2[r] = pd.to_numeric(df_selected_2[r], errors="coerce").clip(0, 100)
# df_selected_2[TIME_COL] = pd.to_numeric(df_selected_2[TIME_COL], errors="coerce")

# # 1분 미만(또는 NaN) 제외
# df_selected_2 = df_selected_2[df_selected_2[TIME_COL].ge(MIN_USED_TIME)]

# # GPU used / not used
# gpu_used_data = df_selected_2[df_selected_2["gpu_allocated(group)"] != "0 GPU"]
# gpu_not_used_data = df_selected_2[df_selected_2["gpu_allocated(group)"] == "0 GPU"]

# def weighted_cdf_props(values, weights, thresholds):
#     """
#     values: utilization series (0~100)
#     weights: runtime(minute) series
#     return: CDF proportions at thresholds (runtime-weighted)
#     """
#     v = pd.to_numeric(values, errors="coerce")
#     w = pd.to_numeric(weights, errors="coerce")

#     mask = v.notna() & w.notna() & (w > 0)
#     v = v[mask].to_numpy()
#     w = w[mask].to_numpy()

#     if len(v) == 0:
#         return [np.nan] * len(thresholds)

#     wsum = w.sum()
#     return [w[v <= t].sum() / wsum for t in thresholds]

# # -----------------------------
# # Runtime-weighted CDF: Non-GPU vs GPU-used
# # -----------------------------
# wprops_5_gpu_used = {}
# wprops_5_gpu_not_used = {}

# for r in resources_without_gpu:
#     wprops_5_gpu_used[r] = weighted_cdf_props(gpu_used_data[r], gpu_used_data[TIME_COL], thresholds_5)
#     wprops_5_gpu_not_used[r] = weighted_cdf_props(gpu_not_used_data[r], gpu_not_used_data[TIME_COL], thresholds_5)

# # GPU는 gpu_used만
# wprops_5_gpu_used["GPU"] = weighted_cdf_props(gpu_used_data["GPU"], gpu_used_data[TIME_COL], thresholds_5)

# # -----------------------------
# # (선택) 그룹별 runtime-weighted CDF
# # -----------------------------
# groups_sorted = np.sort(gpu_used_data["gpu_allocated(group)"].unique())
# wprops_5_by_gpu_group = {g: {} for g in groups_sorted}

# for g in groups_sorted:
#     gd = gpu_used_data[gpu_used_data["gpu_allocated(group)"] == g]
#     for r in resources:
#         wprops_5_by_gpu_group[g][r] = weighted_cdf_props(gd[r], gd[TIME_COL], thresholds_5)

# # -----------------------------
# # Quick sanity check (@100 should be 1.0 if data exists)
# # -----------------------------
# print("[CHECK] runtime-weighted CDF @100")
# for r in resources_without_gpu:
#     print("Non-GPU", r, wprops_5_gpu_not_used[r][-1], "| GPU-used", r, wprops_5_gpu_used[r][-1])
# print("GPU-used GPU", wprops_5_gpu_used["GPU"][-1])
# for g in groups_sorted:
#     print(g, "GPU @100:", wprops_5_by_gpu_group[g]["GPU"][-1])


In [ ]:
_mf_capture_cell(14)


Runtime-weighted P50/P90 + ≤5/≤10 (요약표용)


In [ ]:
_MF_CELL=16
# def weighted_quantile(values, weights, q):
#     """
#     q in [0,1], runtime-weighted quantile
#     """
#     v = pd.to_numeric(values, errors="coerce")
#     w = pd.to_numeric(weights, errors="coerce")
#     mask = v.notna() & w.notna() & (w > 0)
#     v = v[mask].to_numpy()
#     w = w[mask].to_numpy()
#     if len(v) == 0:
#         return np.nan
#     order = np.argsort(v)
#     v = v[order]
#     w = w[order]
#     cw = np.cumsum(w)
#     return v[np.searchsorted(cw, q * cw[-1], side="left")]

# def weighted_share_le(values, weights, thr):
#     v = pd.to_numeric(values, errors="coerce")
#     w = pd.to_numeric(weights, errors="coerce")
#     mask = v.notna() & w.notna() & (w > 0)
#     v = v[mask].to_numpy()
#     w = w[mask].to_numpy()
#     if len(v) == 0:
#         return np.nan
#     return w[v <= thr].sum() / w.sum()

# def runtime_weighted_stats(df, col, label):
#     p50 = weighted_quantile(df[col], df[TIME_COL], 0.50)
#     p90 = weighted_quantile(df[col], df[TIME_COL], 0.90)
#     le5 = weighted_share_le(df[col], df[TIME_COL], 5)
#     le10 = weighted_share_le(df[col], df[TIME_COL], 10)

#     # 총 runtime(분)과 유효 표본(컨테이너 수)도 같이
#     valid_mask = df[col].notna() & df[TIME_COL].notna() & (df[TIME_COL] > 0)
#     total_runtime = df.loc[valid_mask, TIME_COL].sum()
#     n = valid_mask.sum()

#     print(f"{label:10s} {col:6s} | n={n:,} | runtime_sum(min)={total_runtime:,.0f} | "
#           f"wP50={p50:.1f} | wP90={p90:.1f} | w<=5%={le5:.2%} | w<=10%={le10:.2%}")

# for r in ["CPU","Memory"]:
#     runtime_weighted_stats(gpu_not_used_data, r, "Non-GPU")
#     runtime_weighted_stats(gpu_used_data, r, "GPU-used")
# runtime_weighted_stats(gpu_used_data, "GPU", "GPU-used")


In [ ]:
_mf_capture_cell(16)


In [ ]:
_MF_CELL=17
# import numpy as np
# import matplotlib.pyplot as plt

# fig, axes = plt.subplots(1, 3, figsize=(30, 8))
# resources = ["CPU", "Memory", "GPU"]

# # 색상/마커 설정
# colors = ['tab:blue','tab:orange','tab:green','tab:red','tab:purple','tab:brown','tab:pink']
# markers = ['o'] * len(colors)

# for idx, resource_to_plot in enumerate(resources):
#     ax = axes[idx]

#     # --- Non-GPU (0 GPU) ---
#     if resource_to_plot != "GPU":
#         ax.plot(
#             thresholds_5,
#             wprops_5_gpu_not_used[resource_to_plot],
#             label='Non-GPU (runtime-weighted)',
#             color=colors[0],
#             marker=markers[0],
#             markersize=10,
#             linewidth=3
#         )

#     # --- GPU-used 평균 ---
#     ax.plot(
#         thresholds_5,
#         wprops_5_gpu_used[resource_to_plot],
#         label='GPU-Enabled AVG. (runtime-weighted)',
#         color=colors[1],
#         marker=markers[1],
#         markersize=10,
#         linewidth=3
#     )

#     # --- GPU 그룹별 ---
#     for i, grp in enumerate(groups_sorted, start=2):
#         props = wprops_5_by_gpu_group[grp][resource_to_plot]
#         ax.plot(
#             thresholds_5,
#             props,
#             label=grp,
#             color=colors[i % len(colors)],
#             marker=markers[i % len(markers)],
#             markersize=5,
#             linewidth=1,
#             linestyle='--',
#             alpha=0.8
#         )

#     ax.set_title(f"{resource_to_plot}", fontsize=35)
#     ax.set_xlabel("Utilization (%)", fontsize=35)
#     ax.set_xlim(0, 100)
#     ax.set_ylim(0, 1.2)  # ✅ CDF는 0~1 권장
#     ax.set_xticks(range(0, 101, 20))
#     ax.tick_params(axis='x', labelsize=35)
#     ax.tick_params(axis='y', labelsize=35)

#     # --- y축 레이블 및 눈금 설정 ---
#     if idx == 0:
#         ax.set_ylabel("Cumulative Fraction of Runtime", fontsize=30)  # ✅ runtime-weighted
#         ax.legend(loc='lower right', fontsize=24)
#     else:
#         ax.set_yticklabels([])

#     ax.grid(True)

# plt.tight_layout()
# plt.show()


In [ ]:
_mf_capture_cell(17)


## 0409_Resource Usage of Containers with Unused GPUs


In [ ]:
_MF_CELL=19
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import numpy as np
import pandas as pd

# ---- 데이터 ----
df = df_selected_2.copy()

# ===== 1) GPU util 컬럼 자동 선택 =====
gpu_util_col = None
for cand in ["GPU", "gpu_util(%)", "gpu_util", "gpu_util_avg(%)", "gpu_util_max(%)"]:
    if cand in df.columns:
        gpu_util_col = cand
        break
if gpu_util_col is None:
    raise KeyError("GPU utilization column not found. (expected one of: GPU, gpu_util(%), gpu_util, gpu_util_avg(%), gpu_util_max(%))")

# ===== 2) 필요한 컬럼 numeric 처리 =====
EPS = 1.0
df["used_time(minute)"] = pd.to_numeric(df["used_time(minute)"], errors="coerce").replace([0, -1], np.nan).fillna(EPS).clip(lower=EPS)
df["CPU"] = pd.to_numeric(df["CPU"], errors="coerce").fillna(0)
df[gpu_util_col] = pd.to_numeric(df[gpu_util_col], errors="coerce").fillna(0)

# ===== 3) 필터: GPU-Enabled & GPU util == 0 =====
# GPU-Enabled: gpu_allocated(group) != "0 GPU" (혹은 "0.0_GPU" 같은 포맷 변형도 커버)
gcol = "gpu_allocated(group)"
if gcol not in df.columns:
    raise KeyError("gpu_allocated(group) column not found.")

g = df[gcol].astype(str).str.strip()
is_non_gpu = g.isin(["0 GPU", "0.0_GPU", "0_GPU", "0gpu", "0 gpu"])
is_gpu_enabled = ~is_non_gpu

# GPU util == 0 (부동소수 노이즈 대비)
is_gpu_zero = df[gpu_util_col].abs() <= 0  # 필요하면 <= 1e-9 로 완화

df = df[is_gpu_enabled & is_gpu_zero].copy()
print(f"[Filtered] GPU-Enabled & {gpu_util_col}==0 : n_rows={len(df)} | n_containers={df['filename'].nunique() if 'filename' in df.columns else 'NA'}")

# ---- 스타일 ----
sns.set(style="white")  # grid 제거
fig, axes = plt.subplots(1, 2, figsize=(30, 6))

# (1) Run Time (로그 스케일)
sns.violinplot(
    y=df["used_time(minute)"],
    ax=axes[0],
    inner="box", scale="width", cut=0, bw_adjust=0.6, log_scale=(False, True)
)
axes[0].set_ylabel("Run Time (minute)", fontsize=35)
axes[0].tick_params(axis="y", labelsize=35)
axes[0].set_title("Run Time Distribution (GPU-Enabled, GPU util = 0)", fontsize=35)
axes[0].set_xlabel("")
axes[0].grid(False)
axes[0].yaxis.set_major_locator(mtick.LogLocator(base=10))
axes[0].yaxis.set_minor_locator(mtick.LogLocator(base=10, subs=range(2,10)))
axes[0].yaxis.set_major_formatter(mtick.LogFormatterMathtext(base=10))

# (2) CPU (선형)
sns.violinplot(
    y=df["CPU"],
    ax=axes[1],
    inner="box", scale="width", cut=0, bw_adjust=0.6
)
axes[1].set_ylabel("CPU Usage (%)", fontsize=35)
axes[1].tick_params(axis="y", labelsize=35)
axes[1].set_title("CPU Usage Distribution (GPU-Enabled, GPU util = 0)", fontsize=35)
axes[1].set_xlabel("")
axes[1].grid(False)

plt.tight_layout()
plt.show()

# Export the original-code plot inputs for independent verification.
if "_MF_CAPTURE" in globals():
    df[["container_id",gcol,gpu_util_col,"CPU","used_time(minute)"]].to_csv(_MF_CAPTURE/"cell19_figure8_plot_inputs.csv",index=False)
figure8_summary=df[["used_time(minute)","CPU"]].describe(percentiles=[.25,.5,.75,.9,.99]).T
figure8_summary["zero_n"]=df[["used_time(minute)","CPU"]].eq(0).sum()
figure8_allocation_counts=df[gcol].value_counts().sort_index().to_frame("count")
figure8_runtime_violin_vertices=pd.DataFrame(axes[0].collections[0].get_paths()[0].vertices,columns=["x","y"])
figure8_cpu_violin_vertices=pd.DataFrame(axes[1].collections[0].get_paths()[0].vertices,columns=["x","y"])


In [ ]:
_mf_capture_cell(19)


In [ ]:
_MF_CELL=20



In [ ]:
_mf_capture_cell(20)


In [ ]:
_MF_CELL=21



In [ ]:
_mf_capture_cell(21)


In [ ]:
(_MF_OUTPUT/'execution_scope.json').write_text(_mf_json.dumps({'mode':_MF_MODE,'source_notebook':'04_Analysis/section4/Run02_Utilization.ipynb','excluded_original_code_cells':[],'status':'EXECUTED_SCOPE_COMPLETE'},indent=2,ensure_ascii=False))
